In [1]:
import numpy as np
#0 is wall, 1 is accessible
maze = np.array([
    [0,1,0,0,0,0,1,0],
    [0,1,0,1,1,0,1,1],
    [1,1,1,1,0,0,0,1],
    [0,0,1,0,0,1,1,1],
    [0,1,1,1,1,1,0,0],
    [0,1,0,0,0,1,1,0],
    [1,1,1,1,1,0,1,1],
    [0,1,0,0,1,1,1,0],
])

start = (0, 1)
goal = (6, 7)

In [2]:
actions = {
    "U": (-1, 0),
    "D": (1, 0),
    "L": (0, -1),
    "R": (0, 1),
}

gamma = 1.0
step_reward = -1

Definition of transition function

In [3]:
def step(state, action):
    if state == goal:
        return goal, 0, True

    di, dj = actions[action]
    ni, nj = state[0] + di, state[1] + dj

    if (
        ni < 0 or ni >= maze.shape[0]
        or nj < 0 or nj >= maze.shape[1]
        or maze[ni, nj] == 0
    ):
        next_state = state
    else:
        next_state = (ni, nj)

    done = next_state == goal
    reward = 0 if done else step_reward
    return next_state, reward, done

In [4]:
def value_iteration(theta=1e-6, max_iter=1000):
    V = np.zeros_like(maze, dtype=float)#initialize value function matrix

    for it in range(max_iter):
        delta = 0
        V_new = V.copy()

        for i in range(maze.shape[0]):
            for j in range(maze.shape[1]):
                s = (i, j)

                if maze[i, j] == 0:
                    continue

                if s == goal:
                    V_new[i, j] = 0
                    continue

                q_values = []

                for a in actions:
                    ns, r, done = step(s, a)
                    q = r + gamma * V[ns]
                    q_values.append(q)

                V_new[i, j] = max(q_values)#Bellman optimal operator
                delta = max(delta, abs(V_new[i, j] - V[i, j]))

        V = V_new

        if delta < theta:
            break

    return V, it + 1

In [5]:
def extract_policy(V):
    policy = np.full(maze.shape, " ", dtype=object)

    for i in range(maze.shape[0]):
        for j in range(maze.shape[1]):
            s = (i, j)

            if maze[i, j] == 0:
                policy[i, j] = "#"
                continue

            if s == goal:
                policy[i, j] = "G"
                continue

            best_a = None
            best_q = -float("inf")

            for a in actions:
                ns, r, done = step(s, a)
                q = r + gamma * V[ns]

                if q > best_q:
                    best_q = q
                    best_a = a

            policy[i, j] = best_a

    return policy

In [6]:
V, n_iter = value_iteration()
policy = extract_policy(V)

print("Iterations:", n_iter)
print(V)
print(policy)

Iterations: 12
[[  0. -11.   0.   0.   0.   0. -10.   0.]
 [  0. -10.   0. -10. -11.   0.  -9.  -8.]
 [-10.  -9.  -8.  -9.   0.   0.   0.  -7.]
 [  0.   0.  -7.   0.   0.  -4.  -5.  -6.]
 [  0.  -7.  -6.  -5.  -4.  -3.   0.   0.]
 [  0.  -8.   0.   0.   0.  -2.  -1.   0.]
 [ -8.  -7.  -6.  -5.  -4.   0.   0.   0.]
 [  0.  -8.   0.   0.  -3.  -2.  -1.   0.]]
[['#' 'D' '#' '#' '#' '#' 'D' '#']
 ['#' 'D' '#' 'D' 'L' '#' 'R' 'D']
 ['R' 'R' 'D' 'L' '#' '#' '#' 'D']
 ['#' '#' 'D' '#' '#' 'D' 'L' 'L']
 ['#' 'R' 'R' 'R' 'R' 'D' '#' '#']
 ['#' 'U' '#' '#' '#' 'R' 'D' '#']
 ['R' 'R' 'R' 'R' 'D' '#' 'R' 'G']
 ['#' 'U' '#' '#' 'R' 'R' 'U' '#']]


Now we play RTDP according to the book.

In [7]:
import random
from collections import defaultdict

def rtdp(num_episodes=500, max_steps=200, epsilon=0.1):
    V = np.zeros_like(maze, dtype=float)
    visits = np.zeros_like(maze, dtype=int)

    for ep in range(num_episodes):
        s = start

        for t in range(max_steps):
            if s == goal:
                break

            i, j = s
            visits[i, j] += 1

            # Bellman backup only for current state
            q_values = {}
            for a in actions:
                ns, r, done = step(s, a)
                q_values[a] = r + gamma * V[ns]

            V[i, j] = max(q_values.values())

            # choose action: epsilon-greedy
            if random.random() < epsilon:
                a = random.choice(list(actions.keys()))
            else:
                a = max(q_values, key=q_values.get)

            ns, r, done = step(s, a)
            s = ns

            if done:
                break

    return V, visits

In [8]:
V_rtdp, visits = rtdp(num_episodes=1000, epsilon=0.2)
policy_rtdp = extract_policy(V_rtdp)

print(V_rtdp)
print(policy_rtdp)
print(visits)

[[  0. -11.   0.   0.   0.   0.  -4.   0.]
 [  0. -10.   0. -10. -10.   0.  -4.  -5.]
 [-10.  -9.  -8.  -9.   0.   0.   0.  -4.]
 [  0.   0.  -7.   0.   0.  -4.  -5.  -4.]
 [  0.  -7.  -6.  -5.  -4.  -3.   0.   0.]
 [  0.  -8.   0.   0.   0.  -2.  -1.   0.]
 [ -8.  -7.  -6.  -5.  -4.   0.   0.   0.]
 [  0.  -7.   0.   0.  -3.  -2.  -1.   0.]]
[['#' 'D' '#' '#' '#' '#' 'U' '#']
 ['#' 'D' '#' 'D' 'U' '#' 'U' 'D']
 ['R' 'R' 'D' 'L' '#' '#' '#' 'D']
 ['#' '#' 'D' '#' '#' 'D' 'L' 'U']
 ['#' 'R' 'R' 'R' 'R' 'D' '#' '#']
 ['#' 'U' '#' '#' '#' 'R' 'D' '#']
 ['R' 'R' 'R' 'R' 'D' '#' 'R' 'G']
 ['#' 'U' '#' '#' 'R' 'R' 'U' '#']]
[[   0 1268    0    0    0    0    7    0]
 [   0 1238    0   25   16    0    7   10]
 [  91 1272 1297   82    0    0    0    5]
 [   0    0 1254    0    0   85   10    6]
 [   0   72 1243 1291 1287 1272    0    0]
 [   0   17    0    0    0 1249 1256    0]
 [  13   23   14   13   10    0 1202    0]
 [   0   14    0    0   11   18   78    0]]


Finally for Policy iteration. We first try to solve value function by using the matrix inversion: $\mathbf{v}= (\mathbf{I}-\gamma \mathbf{T})^{-1}r$


In [9]:
def build_policy_matrices(policy):
    free_states = [s for s in state_to_idx]
    n = len(free_states)

    T = np.zeros((n, n))
    r = np.zeros(n)

    for s in free_states:
        i = state_to_idx[s]

        if s == goal:
            T[i, i] = 1.0
            r[i] = 0.0
            continue

        a = policy[s]
        ns, reward, done = step(s, a)

        j = state_to_idx[ns]
        T[i, j] = 1.0
        r[i] = reward

    return T, r


def policy_evaluation_direct(policy, gamma=0.99):
    T, r = build_policy_matrices(policy)
    n = len(r)

    A = np.eye(n) - gamma * T
    v = np.linalg.solve(A, r)

    V = np.full(maze.shape, np.nan)
    for s, idx in state_to_idx.items():
        V[s] = v[idx]

    return V

In [10]:
free_states = [(i, j) for i in range(maze.shape[0])
               for j in range(maze.shape[1])
               if maze[i, j] != 0]

state_to_idx = {s: idx for idx, s in enumerate(free_states)}
idx_to_state = {idx: s for s, idx in state_to_idx.items()}

In [11]:
import random

def random_policy():
    policy = {}

    for s in free_states:
        if s == goal:
            policy[s] = None
        else:
            policy[s] = random.choice(list(actions.keys()))

    return policy

In [12]:
policy = random_policy()
V_pi = policy_evaluation_direct(policy, gamma=0.99)

print(V_pi)

[[  nan -100.   nan   nan   nan   nan -100.   nan]
 [  nan -100.   nan -100. -100.   nan -100. -100.]
 [-100. -100. -100. -100.   nan   nan   nan -100.]
 [  nan   nan -100.   nan   nan -100. -100. -100.]
 [  nan -100. -100. -100. -100. -100.   nan   nan]
 [  nan -100.   nan   nan   nan -100. -100.   nan]
 [-100. -100. -100. -100. -100.   nan -100.    0.]
 [  nan -100.   nan   nan -100. -100. -100.   nan]]


In [13]:
arrow_map = {
    "U": "↑",
    "D": "↓",
    "L": "←",
    "R": "→",
    None: "G"
}

def print_policy(policy):
    display = np.full(maze.shape, "#", dtype=object)

    for i in range(maze.shape[0]):
        for j in range(maze.shape[1]):

            if maze[i, j] == 0:
                display[i, j] = "#"
                continue

            s = (i, j)

            if s == goal:
                display[i, j] = "G"
            else:
                display[i, j] = arrow_map[policy[s]]

    for row in display:
        print(" ".join(row))

In [14]:
policy = random_policy()
print_policy(policy)

# → # # # # → #
# → # ↓ → # ↓ ←
↑ ← ← ↓ # # # →
# # ↓ # # ↑ ← ←
# ↓ ↑ ↑ ↓ ← # #
# ↑ # # # ↓ → #
→ ↑ → ↑ ↑ # ↑ G
# ↑ # # ↑ ↓ → #


In [15]:
def policy_improvement(V, policy):
    stable = True

    for s in free_states:

        if s == goal:
            continue

        best_action = None
        best_q = -np.inf

        for a in actions:

            ns, r, done = step(s, a)

            q = r + gamma * V[ns]

            if q > best_q:
                best_q = q
                best_action = a

        if best_action != policy[s]:
            stable = False

        policy[s] = best_action

    return policy, stable

In [16]:
def policy_iteration():

    policy = random_policy()

    iteration = 0

    while True:

        iteration += 1

        V = policy_evaluation_direct(policy)

        policy, stable = policy_improvement(V, policy)

        print(f"\nPI iteration {iteration}")
        print_policy(policy)

        if stable:
            break

    return V, policy

In [17]:
policy_iteration()


PI iteration 1
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↑ ↑ ↑
# ↑ → ↑ ↑ ← # #
# ↑ # # # ↑ ↓ #
↑ ↑ ↑ ↑ ↑ # → G
# ↑ # # ↑ ↑ ↑ #

PI iteration 2
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↑ ↑ ↑
# ↑ ↑ ↑ ↑ ↑ # #
# ↑ # # # → ↓ #
↑ ↑ ↑ ↑ ↑ # → G
# ↑ # # ↑ → ↑ #

PI iteration 3
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↑ ↑ ↑
# ↑ ↑ ↑ ↑ ↓ # #
# ↑ # # # → ↓ #
↑ ↑ ↑ ↑ ↑ # → G
# ↑ # # → → ↑ #

PI iteration 4
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↓ ↑ ↑
# ↑ ↑ ↑ → ↓ # #
# ↑ # # # → ↓ #
↑ ↑ ↑ ↑ ↓ # → G
# ↑ # # → → ↑ #

PI iteration 5
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↓ ← ↑
# ↑ ↑ → → ↓ # #
# ↑ # # # → ↓ #
↑ ↑ ↑ → ↓ # → G
# ↑ # # → → ↑ #

PI iteration 6
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↑
# # ↑ # # ↓ ← ←
# ↑ → → → ↓ # #
# ↑ # # # → ↓ #
↑ ↑ → → ↓ # → G
# ↑ # # → → ↑ #

PI iteration 7
# ↑ # # # # ↑ #
# ↑ # ↑ ↑ # ↑ ↑
↑ ↑ ↑ ↑ # # # ↓
# # ↓ # # ↓ ← ←
# → → → → ↓ # #
# ↑ # # # → ↓ #
↑ → → → ↓ # → G
# ↑ # # 

(array([[         nan, -10.46617457,          nan,          nan,
                  nan,          nan,  -9.5617925 ,          nan],
        [         nan,  -9.5617925 ,          nan,  -9.5617925 ,
         -10.46617457,          nan,  -8.64827525,  -7.72553056],
        [ -9.5617925 ,  -8.64827525,  -7.72553056,  -8.64827525,
                  nan,          nan,          nan,  -6.79346521],
        [         nan,          nan,  -6.79346521,          nan,
                  nan,  -3.940399  ,  -4.90099501,  -5.85198506],
        [         nan,  -6.79346521,  -5.85198506,  -4.90099501,
          -3.940399  ,  -2.9701    ,          nan,          nan],
        [         nan,  -7.72553056,          nan,          nan,
                  nan,  -1.99      ,  -1.        ,          nan],
        [ -7.72553056,  -6.79346521,  -5.85198506,  -4.90099501,
          -3.940399  ,          nan,   0.        ,  -0.        ],
        [         nan,  -7.72553056,          nan,          nan,
          -2.9701 